In [1]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline


In [2]:

# =========================
# 1) Load prepared numeric data
# =========================
df = pd.read_csv("../fraud_preprocessing/fraud_prepared_numeric.csv")

target = "fraud"
X = df.drop(columns=[target])
y = df[target].astype(int)


In [3]:

# =========================
# 2) Split columns:
#    - binary columns: keep as-is
#    - continuous columns: scale
# =========================
binary_cols = [c for c in X.columns if set(X[c].dropna().unique()).issubset({0, 1})]
continuous_cols = [c for c in X.columns if c not in binary_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), continuous_cols),
        ("binary", "passthrough", binary_cols),
    ],
    remainder="drop"
)


In [4]:

# =========================
# 3) Base model for wrapper selection
#    class_weight='balanced' is important for fraud imbalance
# =========================
base_model = LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    max_iter=2000,
    random_state=42
)


In [5]:

# =========================
# 4) RFECV: recursive feature elimination with CV
#    scoring='average_precision' is better than accuracy for fraud problems
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

selector = RFECV(
    estimator=base_model,
    step=2,                       # remove 2 features at a time
    cv=cv,
    scoring="average_precision",  # better for imbalanced fraud label
    min_features_to_select=5,
    n_jobs=-1
)

pipe = Pipeline([
    ("prep", preprocess),
    ("rfecv", selector)
])

# Fit selector
pipe.fit(X, y)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('rfecv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('binary', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers conta

In [6]:

# =========================
# 5) Recover selected feature names
# =========================
feature_names_after_prep = pipe.named_steps["prep"].get_feature_names_out()
support_mask = pipe.named_steps["rfecv"].support_
ranking = pipe.named_steps["rfecv"].ranking_

selected_features = [
    name.replace("scale__", "").replace("binary__", "")
    for name, keep in zip(feature_names_after_prep, support_mask)
    if keep
]

ranking_df = pd.DataFrame({
    "feature": [
        name.replace("scale__", "").replace("binary__", "")
        for name in feature_names_after_prep
    ],
    "rank": ranking
}).sort_values(["rank", "feature"])

# Remove duplicates just in case
selected_features = list(dict.fromkeys(selected_features))


In [7]:

# =========================
# 6) Save reduced dataset
# =========================
OUT_DIR = Path("../wrapper")
OUT_DIR.mkdir(parents=True, exist_ok=True)

selected_df = df[selected_features + [target]].copy()
selected_df.to_csv("../wrapper/fraud_selected_features_rfecv.csv", index=False)


In [8]:

# =========================
# 7) Print results
# =========================
print("Original number of features :", X.shape[1])
print("Selected number of features :", len(selected_features))
print("\nSelected features:")
for f in selected_features:
    print("-", f)

print("\nTop-ranked features:")
print(ranking_df.head(30).to_string(index=False))

print("\nReduced dataset saved as: fraud_selected_features_rfecv.csv")

Original number of features : 55
Selected number of features : 17

Selected features:
- Latitude
- order_weekday
- Category Name__freq
- Product Image__freq
- Product Name__freq
- order_is_weekend
- Customer Zipcode_was_missing
- Customer Country_EE. UU.
- Customer Country_Puerto Rico
- Customer Segment_Consumer
- Customer Segment_Corporate
- Department Name_Book Shop
- Department Name_Discs Shop
- Department Name_Footwear
- Department Name_Health and Beauty 
- Department Name_Outdoors
- Shipping Mode_First Class

Top-ranked features:
                           feature  rank
               Category Name__freq     1
          Customer Country_EE. UU.     1
      Customer Country_Puerto Rico     1
         Customer Segment_Consumer     1
        Customer Segment_Corporate     1
      Customer Zipcode_was_missing     1
         Department Name_Book Shop     1
        Department Name_Discs Shop     1
          Department Name_Footwear     1
Department Name_Health and Beauty      1
        